# model_test — 기본 피처 확장: 선형 + LightGBM (로컬 CPU)

설정은 `config/experiment.yaml`의 `BASE_FEATURES.model_input`(linear, lightgbm). 계산 코드는 `src/evaluation/base_model_runs.py`·`model_test.py`, 이 노트북은 함수만 부른다. **Run All**로 실행한다.

- **입력 세트 6개**: 기본 / +H1 / +H3 / +H1+H3 / +H2 / +H2+H4 (`src/evaluation/base_model_runs.input_sets`).
- **결측**: 관측값으로 매달 변환한 뒤 빈 칸 0(= 그 달 평균), 학습·예측 모두. **교수님 지침의 예외(2026-11-21 사용자 결정, 확인 전)**. 근거 Chen·McCoy 2024 JFE, GKX 2020. 목표값은 채우지 않음. 목표월 2001-01·02는 학습에서 뺌(시장 변동성 12개월 창).
- **중간 저장·재시작**: 단위가 끝날 때마다 `<결과 위치>/<AUTHOR>/<RUN_NAME>_parts/`에 저장. 끊기면 커널 재시작 후 **Run All**만 다시 하면 저장된 단위는 건너뛰고 이어간다. 설정·코드·입력이 저장 때와 다르면 섞이지 않게 멈춘다(그때는 RUN_NAME을 바꿔 새로 시작). 결과 폴더 저장도 한 번만 한다.
- **선형**: OLS·Ridge × 목표값 윈저·월별 z × 손실 MSE·Huber①·Huber② = 12설정, 입력 변환 순위·z·윈저z 3가지(변환마다 결과 폴더 하나). 기준선 변환은 세트 6(전체)에서 12설정 중 1등 설정의 바깥 검증 평균 IC가 가장 높은 것(실행 전 고정).
- **LightGBM**: 회귀 4(윈저·z × MSE·Huber②) + 순위회귀 1, 깊이 3, 학습률 0.01·0.05, 트리 수 해마다 선택과 500 고정(같은 학습, hparams 끝 `_t500`), 입력 순위. 종류마다 결과 폴더 하나.

In [ ]:
import sys, platform, os
print('python:', sys.version.split()[0], platform.platform())
print('CPU 개수:', os.cpu_count())

## 1. 설정

In [ ]:
import os, sys
from datetime import datetime
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt
from src.evaluation import base_model_runs as bmr
from src.features.h3 import compute_h3_sigma_grid

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_no_test(config)

AUTHOR = 'jiun'                 # 결과 폴더 outputs/model_results/<AUTHOR>/ (영문·숫자·_·-만)
RUN_NAME = 'basefeat_v1'        # 같은 이름으로 다시 Run All 하면 저장된 단위를 이어 씀. 조건을 바꿔 새로 돌릴 때만 바꾼다
LINEAR_WORKERS = 6              # 선형: Fold를 나눠 돌릴 프로세스 수(CPU 8개 기준)
LGB_THREADS = None              # LightGBM 스레드 수(None = 라이브러리 기본)
OUT_ROOT = None                 # 기본 ROOT/outputs/model_results
STARTED = datetime.now()
PARTS = bmr.parts_dir(ROOT, config, author=AUTHOR, run_name=RUN_NAME, out_root=OUT_ROOT)
print('중간 저장 폴더:', PARTS)
print('이미 저장된 단위:', len(list(PARTS.glob('*.pkl'))))

In [ ]:
# H3 σ 후보(λ 10개)는 한 번만 계산해 전처리에 같이 쓴다
panel = pd.read_parquet(ROOT / config['PATHS']['panel'], columns=['stock_id', 'month', 'return', 'eligible'])
eligibility = pd.read_parquet(ROOT / config['PATHS']['eligibility'])
H3_GRID = compute_h3_sigma_grid(panel, eligibility)
del panel, eligibility
print('H3 σ 격자:', H3_GRID.shape)

## 2. 선형 (변환 3가지 × 단위 6개 = 18단위)

In [ ]:
LIN = config['BASE_FEATURES']['model_input']['linear']
LINEAR_PREP, LINEAR_RESULTS = {}, {}
for t in LIN['transforms_compared']:
    LINEAR_PREP[t] = bmr.prepare(ROOT, config, 'linear', input_transform=t, h3_grid=H3_GRID)
    fp = bmr.fingerprint(ROOT, config, 'linear', t)
    LINEAR_RESULTS.update(bmr.run_units(bmr.linear_units(LINEAR_PREP[t], config, workers=LINEAR_WORKERS), PARTS, fp))

In [ ]:
# 변환마다 결과 폴더 하나로 저장(이미 저장했으면 건너뜀)
for t, prep in LINEAR_PREP.items():
    print(bmr.finalize_linear(ROOT, config, prep, LINEAR_RESULTS, author=AUTHOR, run_name=RUN_NAME, started=STARTED,
                              parts=PARTS, out_root=OUT_ROOT))
del LINEAR_PREP, LINEAR_RESULTS

## 3. LightGBM (세트 6 × 종류 2 × 학습률 2 = 24단위)

In [ ]:
LGB_PREP = bmr.prepare(ROOT, config, 'boosting', input_transform='rank', h3_grid=H3_GRID)
fp = bmr.fingerprint(ROOT, config, 'boosting', 'rank')
LGB_RESULTS = bmr.run_units(bmr.boosting_units(LGB_PREP, config, 'lightgbm', n_threads=LGB_THREADS), PARTS, fp)

In [ ]:
for r in bmr.finalize_boosting(ROOT, config, LGB_PREP, LGB_RESULTS, author=AUTHOR, run_name=RUN_NAME, started=STARTED,
                               parts=PARTS, out_root=OUT_ROOT):
    print(r)